
# 11 — Recovery Validation  
## External Outcome Validation of the Main 5-Variable POSet

This notebook validates the main POSet results against GDP recovery outcomes.

It runs after:

```text
10_Epsilon_Margin_POSet_Robustness_2002_5Var.ipynb
```

## Final decisions reflected here

- Recovery is **not** an ordering variable.
- Recovery is used only as an external outcome / validation variable.
- Main POSet result = Notebook 08 profile-level POSet.
- Epsilon-margin = robustness/sensitivity only.
- Final ordering variables remain **5**.
- WGI/governance is not a final ordering variable.

## Validation question

After constructing the structural POSet from pre-shock variables, we ask:

```text
Do countries in stronger structural positions show faster GDP recovery after the shock?
```

This is a validation/interpretation exercise, not a causal proof.

## What gets compared

The notebook compares GDP recovery years across:

1. Pareto/frontier vs non-frontier countries;
2. POSet layers;
3. profile groups;
4. epsilon-margin frontier vs non-frontier countries, as robustness only;
5. mismatch cases:
   - strong structure but slow recovery;
   - weak structure but fast recovery.

## Main outputs

- `profile_recovery_validation_summary.csv`
- `profile_layer_recovery_validation_summary.csv`
- `pareto_recovery_validation_table.csv`
- `epsilon_margin_recovery_validation_summary.csv`
- `recovery_interpretation_cases.csv`
- `recovery_validation_takeaway_table.csv`
- `working_main_profile_recovery_validation_summary.csv`
- `working_main_epsilon_margin_recovery_validation_summary.csv`
- `report_ready_recovery_validation_paragraphs.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
EPSILON_MARGIN_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "11_Recovery_Validation"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Recovery_Validation"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Profile folder:", PROFILE_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_202059
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Profile folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Recovery_Validation


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

PROFILE_COUNTRY_MAP_FILE = PROFILE_DIR / "profile_country_map_with_layers.csv"
PROFILE_LAYER_SUMMARY_FILE = PROFILE_DIR / "profile_layer_summary.csv"
PROFILE_QUALITY_FILE = PROFILE_DIR / "profile_poset_quality_diagnostics.csv"

STRUCTURAL_COMPLETE_CASE_FILE = MASTER_DIR / "structural_snapshot_final5_complete_cases.csv"

EPSILON_MARGIN_COUNTRY_SUMMARY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_country_summary.csv"
EPSILON_MARGIN_RUN_SUMMARY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_run_summary.csv"

REQUIRED_FILES = [
    PROFILE_COUNTRY_MAP_FILE,
    STRUCTURAL_COMPLETE_CASE_FILE,
]

for path in REQUIRED_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

RECOVERY_COL_BY_SHOCK = {
    "2007": "Recovery_2007",
    "2019": "Recovery_2019",
}

RECOVERY_STATUS_COL_BY_SHOCK = {
    "2007": "Recovery_2007_status",
    "2019": "Recovery_2019_status",
}

REFERENCE_EPSILON_MARGIN = 0.10

print("Reference epsilon-margin:", REFERENCE_EPSILON_MARGIN)
print("Recovery columns:", RECOVERY_COL_BY_SHOCK)


Reference epsilon-margin: 0.1
Recovery columns: {'2007': 'Recovery_2007', '2019': 'Recovery_2019'}


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


def safe_mean(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.mean() if len(series) else np.nan


def safe_median(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.median() if len(series) else np.nan


In [4]:

# ------------------------------------------------------
# LOAD PROFILE POSET AND MASTER RECOVERY DATA
# ------------------------------------------------------

profile_country_map = pd.read_csv(PROFILE_COUNTRY_MAP_FILE)
structural = pd.read_csv(STRUCTURAL_COMPLETE_CASE_FILE)

for df_ in [profile_country_map, structural]:
    df_["Country"] = df_["Country"].astype(str).str.strip().str.upper()
    df_["shock_id"] = df_["shock_id"].astype(str)
    df_["baseline_year"] = pd.to_numeric(df_["baseline_year"], errors="coerce").astype(int)

needed_profile_cols = {"Country", "shock_id", "baseline_year", "profile_code", "layer", "is_pareto_frontier"}
missing_profile = needed_profile_cols - set(profile_country_map.columns)

if missing_profile:
    raise ValueError(f"Profile country map missing required columns: {missing_profile}")

# Build a long recovery table from structural snapshot.
structural_recovery_frames = []

for shock_id, recovery_col in RECOVERY_COL_BY_SHOCK.items():
    temp = structural[structural["shock_id"].astype(str).eq(shock_id)].copy()

    if recovery_col not in temp.columns:
        raise ValueError(f"Missing recovery column in structural data: {recovery_col}")

    status_col = RECOVERY_STATUS_COL_BY_SHOCK.get(shock_id)

    temp["recovery_years"] = pd.to_numeric(temp[recovery_col], errors="coerce")
    temp["recovery_status"] = temp[status_col] if status_col in temp.columns else np.nan

    structural_recovery_frames.append(
        temp[
            [
                "Country", "shock_id", "baseline_year",
                "recovery_years", "recovery_status",
            ] + [c for c in FINAL_5_SCORE_COLUMNS if c in temp.columns]
        ]
    )

structural_recovery = pd.concat(structural_recovery_frames, ignore_index=True)

validation_dataset = profile_country_map.merge(
    structural_recovery,
    on=["Country", "shock_id", "baseline_year"],
    how="left",
    suffixes=("", "_structural"),
)

validation_dataset["recovery_years"] = pd.to_numeric(validation_dataset["recovery_years"], errors="coerce")
validation_dataset["layer"] = pd.to_numeric(validation_dataset["layer"], errors="coerce").astype(int)
validation_dataset["is_pareto_frontier"] = validation_dataset["is_pareto_frontier"].astype(bool)

save_table(
    validation_dataset,
    "recovery_validation_country_dataset.csv",
    "Recovery validation country dataset",
    "Country-level profile POSet results merged with GDP recovery outcomes.",
)

print("Validation dataset rows:", len(validation_dataset))
display(validation_dataset.head())


Saved: recovery_validation_country_dataset.csv
Validation dataset rows: 60


,Country,shock_id,baseline_year,profile_code,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels,layer,is_pareto_frontier,recovery_years,recovery_status,debt_capacity_score_0_100_structural,employment_strength_score_0_100_structural,rd_intensity_score_0_100_structural,human_capital_tertiary_score_0_100_structural,energy_security_score_0_100_structural
0,AUT,2007,2007,2-4-5-3-2,16.0000,2,4,5,3,2,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,5,2,False,2.0000,recovered,38.5303,80.6046,68.7303,33.2757,22.6201
1,BEL,2007,2007,1-2-4-4-1,12.0000,1,2,4,4,1,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,5,3,False,1.0000,recovered,16.9811,46.5777,48.5360,53.5113,9.6139
2,CAN,2007,2007,3-3-4-5-5,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5,1,True,1.0000,recovered,64.9865,63.6697,50.3901,100.0000,100.0000
3,CZE,2007,2007,4-3-3-1-5,16.0000,4,3,3,1,5,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,5,2,False,5.0000,recovered,76.7627,74.6107,29.3904,0.4466,50.5562
4,DEU,2007,2007,2-1-4-2-3,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5,4,False,2.0000,recovered,40.6157,31.1478,68.2315,30.9850,28.5833


In [5]:

# ------------------------------------------------------
# PROFILE FRONTIER VS NON-FRONTIER VALIDATION
# ------------------------------------------------------

profile_recovery_validation_summary = (
    validation_dataset
    .groupby(["shock_id", "baseline_year", "is_pareto_frontier"])
    .agg(
        countries=("Country", "nunique"),
        mean_recovery_years=("recovery_years", "mean"),
        median_recovery_years=("recovery_years", "median"),
        min_recovery_years=("recovery_years", "min"),
        max_recovery_years=("recovery_years", "max"),
        countries_with_recovery_data=("recovery_years", lambda x: int(pd.to_numeric(x, errors="coerce").notna().sum())),
        country_list=("Country", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
    .sort_values(["shock_id", "is_pareto_frontier"], ascending=[True, False])
)

# Wide comparison: frontier minus non-frontier.
comparison_rows = []

for shock_id, group in profile_recovery_validation_summary.groupby("shock_id"):
    baseline_year = int(group["baseline_year"].iloc[0])

    frontier = group[group["is_pareto_frontier"].eq(True)]
    non_frontier = group[group["is_pareto_frontier"].eq(False)]

    if len(frontier) and len(non_frontier):
        mean_diff = frontier["mean_recovery_years"].iloc[0] - non_frontier["mean_recovery_years"].iloc[0]
        median_diff = frontier["median_recovery_years"].iloc[0] - non_frontier["median_recovery_years"].iloc[0]
    else:
        mean_diff = np.nan
        median_diff = np.nan

    comparison_rows.append({
        "shock_id": shock_id,
        "baseline_year": baseline_year,
        "frontier_mean_minus_nonfrontier_mean_recovery": mean_diff,
        "frontier_median_minus_nonfrontier_median_recovery": median_diff,
        "interpretation": (
            "Negative values mean frontier countries recovered faster on average/median."
            if pd.notna(mean_diff)
            else "Insufficient groups for comparison."
        ),
    })

profile_frontier_recovery_comparison = pd.DataFrame(comparison_rows)

pareto_recovery_validation_table = validation_dataset[
    validation_dataset["is_pareto_frontier"]
].copy().sort_values(["shock_id", "recovery_years", "Country"])

save_table(
    profile_recovery_validation_summary,
    "profile_recovery_validation_summary.csv",
    "Profile recovery validation summary",
    "Recovery summary comparing main POSet frontier vs non-frontier countries.",
)

save_table(
    profile_frontier_recovery_comparison,
    "profile_frontier_recovery_comparison.csv",
    "Profile frontier recovery comparison",
    "Difference between frontier and non-frontier recovery outcomes.",
)

save_table(
    pareto_recovery_validation_table,
    "pareto_recovery_validation_table.csv",
    "Pareto recovery validation table",
    "Country-level recovery outcomes for main POSet frontier countries.",
)

display(profile_recovery_validation_summary)
display(profile_frontier_recovery_comparison)


Saved: profile_recovery_validation_summary.csv
Saved: profile_frontier_recovery_comparison.csv
Saved: pareto_recovery_validation_table.csv


,shock_id,baseline_year,is_pareto_frontier,countries,mean_recovery_years,median_recovery_years,min_recovery_years,max_recovery_years,countries_with_recovery_data,country_list
1,2007,2007,True,8,5.0000,6.0000,1.0000,8.0000,8,"CAN, DNK, EST, FIN, GBR, LUX, SVN, USA"
0,2007,2007,False,17,5.8235,5.0000,0.0000,20.0000,17,"AUT, BEL, CZE, DEU, ESP, FRA, GRC, HUN, IRL, I..."
3,2019,2019,True,13,1.0000,1.0000,0.0000,2.0000,13,"AUS, CAN, CHE, CZE, DEU, DNK, EST, IRL, KOR, L..."
2,2019,2019,False,22,1.2632,1.0000,0.0000,2.0000,19,"AUT, BEL, BGR, COL, ESP, FIN, FRA, GBR, GRC, H..."


,shock_id,baseline_year,frontier_mean_minus_nonfrontier_mean_recovery,frontier_median_minus_nonfrontier_median_recovery,interpretation
0,2007,2007,-0.8235,1.0000,Negative values mean frontier countries recove...
1,2019,2019,-0.2632,0.0000,Negative values mean frontier countries recove...


In [6]:

# ------------------------------------------------------
# POSET LAYER RECOVERY VALIDATION
# ------------------------------------------------------

profile_layer_recovery_validation_summary = (
    validation_dataset
    .groupby(["shock_id", "baseline_year", "layer"])
    .agg(
        countries=("Country", "nunique"),
        mean_recovery_years=("recovery_years", "mean"),
        median_recovery_years=("recovery_years", "median"),
        min_recovery_years=("recovery_years", "min"),
        max_recovery_years=("recovery_years", "max"),
        country_list=("Country", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
    .sort_values(["shock_id", "layer"])
)

# Spearman-style rank correlation using pandas rank correlation.
layer_recovery_trend_rows = []

for shock_id, group in validation_dataset.groupby("shock_id"):
    valid = group[["layer", "recovery_years"]].dropna().copy()

    if len(valid) >= 3 and valid["layer"].nunique() > 1 and valid["recovery_years"].nunique() > 1:
        corr = valid["layer"].corr(valid["recovery_years"], method="spearman")
    else:
        corr = np.nan

    layer_recovery_trend_rows.append({
        "shock_id": shock_id,
        "n_countries": len(valid),
        "layer_recovery_spearman_correlation": corr,
        "interpretation": (
            "Positive means deeper/worse layers tend to have slower recovery; negative means the opposite."
            if pd.notna(corr)
            else "Insufficient variation for trend correlation."
        ),
    })

profile_layer_recovery_trend = pd.DataFrame(layer_recovery_trend_rows)

save_table(
    profile_layer_recovery_validation_summary,
    "profile_layer_recovery_validation_summary.csv",
    "Profile layer recovery validation summary",
    "Recovery summary by POSet layer for the main profile POSet.",
)

save_table(
    profile_layer_recovery_trend,
    "profile_layer_recovery_trend.csv",
    "Profile layer recovery trend",
    "Spearman-style correlation between POSet layer and recovery years.",
)

display(profile_layer_recovery_validation_summary)
display(profile_layer_recovery_trend)


Saved: profile_layer_recovery_validation_summary.csv
Saved: profile_layer_recovery_trend.csv


,shock_id,baseline_year,layer,countries,mean_recovery_years,median_recovery_years,min_recovery_years,max_recovery_years,country_list
0,2007,2007,1,8,5.0000,6.0000,1.0000,8.0000,"CAN, DNK, EST, FIN, GBR, LUX, SVN, USA"
1,2007,2007,2,7,4.8571,5.0000,2.0000,8.0000,"AUT, CZE, ESP, IRL, LTU, NLD, SWE"
2,2007,2007,3,5,5.6000,2.0000,0.0000,15.0000,"BEL, FRA, ITA, LVA, POL"
3,2007,2007,4,3,2.6667,2.0000,1.0000,5.0000,"DEU, HUN, SVK"
4,2007,2007,5,2,14.5000,14.5000,9.0000,20.0000,"GRC, PRT"
5,2019,2019,1,13,1.0000,1.0000,0.0000,2.0000,"AUS, CAN, CHE, CZE, DEU, DNK, EST, IRL, KOR, L..."
6,2019,2019,2,10,1.0000,1.0000,0.0000,2.0000,"AUT, BEL, FIN, GBR, HUN, LTU, NLD, NZL, ROU, SVN"
7,2019,2019,3,8,1.5000,2.0000,0.0000,2.0000,"BGR, ESP, FRA, HRV, LVA, PRT, TUR, ZAF"
8,2019,2019,4,4,1.5000,1.5000,1.0000,2.0000,"COL, GRC, ITA, SVK"


,shock_id,n_countries,layer_recovery_spearman_correlation,interpretation
0,2007,25,0.0698,Positive means deeper/worse layers tend to hav...
1,2019,32,0.3095,Positive means deeper/worse layers tend to hav...


In [7]:

# ------------------------------------------------------
# PROFILE GROUP VALIDATION
# ------------------------------------------------------

profile_group_recovery_summary = (
    validation_dataset
    .groupby(["shock_id", "baseline_year", "profile_code", "layer", "is_pareto_frontier"])
    .agg(
        countries=("Country", "nunique"),
        mean_recovery_years=("recovery_years", "mean"),
        median_recovery_years=("recovery_years", "median"),
        country_list=("Country", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
    .sort_values(["shock_id", "layer", "mean_recovery_years", "profile_code"])
)

save_table(
    profile_group_recovery_summary,
    "profile_group_recovery_summary.csv",
    "Profile group recovery summary",
    "Recovery summary by five-dimensional ordinal profile.",
)

display(profile_group_recovery_summary.head(50))


Saved: profile_group_recovery_summary.csv


,shock_id,baseline_year,profile_code,layer,is_pareto_frontier,countries,mean_recovery_years,median_recovery_years,country_list
9,2007,2007,2-4-5-5-4,1,True,1,1.0000,1.0000,USA
12,2007,2007,3-3-4-5-5,1,True,1,1.0000,1.0000,CAN
24,2007,2007,5-5-3-3-1,1,True,1,2.0000,2.0000,LUX
4,2007,2007,1-4-3-5-5,1,True,1,5.0000,5.0000,GBR
19,2007,2007,4-5-5-4-5,1,True,1,7.0000,7.0000,DNK
11,2007,2007,3-2-5-5-3,1,True,1,8.0000,8.0000,FIN
21,2007,2007,5-4-3-2-4,1,True,1,8.0000,8.0000,SVN
23,2007,2007,5-5-2-5-5,1,True,1,8.0000,8.0000,EST
8,2007,2007,2-4-5-3-2,2,False,1,2.0000,2.0000,AUT
13,2007,2007,3-3-5-4-4,2,False,1,2.0000,2.0000,SWE


In [8]:

# ------------------------------------------------------
# EPSILON-MARGIN RECOVERY VALIDATION — ROBUSTNESS ONLY
# ------------------------------------------------------

if EPSILON_MARGIN_COUNTRY_SUMMARY_FILE.exists():
    eps_margin = pd.read_csv(EPSILON_MARGIN_COUNTRY_SUMMARY_FILE)
    eps_margin["Country"] = eps_margin["Country"].astype(str).str.strip().str.upper()
    eps_margin["shock_id"] = eps_margin["shock_id"].astype(str)
    eps_margin["baseline_year"] = pd.to_numeric(eps_margin["baseline_year"], errors="coerce").astype(int)
    eps_margin["epsilon_margin"] = pd.to_numeric(eps_margin["epsilon_margin"], errors="coerce")

    eps_margin_ref = eps_margin[
        eps_margin["epsilon_margin"].eq(REFERENCE_EPSILON_MARGIN)
    ].copy()

    epsilon_margin_validation_dataset = eps_margin_ref.merge(
        structural_recovery,
        on=["Country", "shock_id", "baseline_year"],
        how="left",
        suffixes=("", "_recovery"),
    )

    epsilon_margin_validation_dataset["recovery_years"] = pd.to_numeric(
        epsilon_margin_validation_dataset["recovery_years"],
        errors="coerce",
    )

    epsilon_margin_recovery_validation_summary = (
        epsilon_margin_validation_dataset
        .groupby(["shock_id", "baseline_year", "epsilon_margin", "is_frontier"])
        .agg(
            countries=("Country", "nunique"),
            mean_recovery_years=("recovery_years", "mean"),
            median_recovery_years=("recovery_years", "median"),
            min_recovery_years=("recovery_years", "min"),
            max_recovery_years=("recovery_years", "max"),
            country_list=("Country", lambda x: ", ".join(sorted(x))),
        )
        .reset_index()
        .sort_values(["shock_id", "is_frontier"], ascending=[True, False])
    )

else:
    epsilon_margin_validation_dataset = pd.DataFrame()
    epsilon_margin_recovery_validation_summary = pd.DataFrame([
        {
            "shock_id": "not_available",
            "baseline_year": np.nan,
            "epsilon_margin": REFERENCE_EPSILON_MARGIN,
            "is_frontier": np.nan,
            "countries": 0,
            "mean_recovery_years": np.nan,
            "median_recovery_years": np.nan,
            "min_recovery_years": np.nan,
            "max_recovery_years": np.nan,
            "country_list": "",
        }
    ])

working_main_epsilon_margin_recovery_validation_summary = epsilon_margin_recovery_validation_summary.copy()
working_main_epsilon_margin_recovery_validation_summary["reference_role"] = "robustness_reference_only_not_main_profile_poset"

save_table(
    epsilon_margin_validation_dataset,
    "epsilon_margin_recovery_validation_dataset.csv",
    "Epsilon-margin recovery validation dataset",
    "Country-level epsilon-margin frontier status merged with recovery outcomes. Robustness only.",
)

save_table(
    epsilon_margin_recovery_validation_summary,
    "epsilon_margin_recovery_validation_summary.csv",
    "Epsilon-margin recovery validation summary",
    "Recovery comparison by epsilon-margin frontier status at reference epsilon.",
)

save_table(
    working_main_epsilon_margin_recovery_validation_summary,
    "working_main_epsilon_margin_recovery_validation_summary.csv",
    "Working main epsilon-margin recovery validation summary",
    "Reference epsilon-margin recovery validation summary. Robustness only.",
)

display(epsilon_margin_recovery_validation_summary)


Saved: epsilon_margin_recovery_validation_dataset.csv
Saved: epsilon_margin_recovery_validation_summary.csv
Saved: working_main_epsilon_margin_recovery_validation_summary.csv


,shock_id,baseline_year,epsilon_margin,is_frontier,countries,mean_recovery_years,median_recovery_years,min_recovery_years,max_recovery_years,country_list
1,2007,2007,0.1000,True,12,4.8333,5.0000,1.0000,8.0000,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S..."
0,2007,2007,0.1000,False,13,6.2308,5.0000,0.0000,20.0000,"AUT, BEL, DEU, ESP, FRA, GRC, HUN, ITA, LVA, N..."
3,2019,2019,0.1000,True,20,1.0000,1.0000,0.0000,2.0000,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F..."
2,2019,2019,0.1000,False,15,1.3571,1.5000,0.0000,2.0000,"BEL, COL, ESP, FRA, GBR, GRC, HRV, ITA, LTU, L..."


In [9]:

# ------------------------------------------------------
# MISMATCH / INTERPRETATION CASES
# ------------------------------------------------------
# Strong structure but slow recovery:
#   - main frontier and recovery worse than median for that shock.
#
# Weak structure but fast recovery:
#   - deepest layer and recovery faster than median for that shock.

case_rows = []

for shock_id, group in validation_dataset.groupby("shock_id"):
    shock_median_recovery = group["recovery_years"].median()
    max_layer = group["layer"].max()

    strong_slow = group[
        group["is_pareto_frontier"]
        & group["recovery_years"].notna()
        & (group["recovery_years"] > shock_median_recovery)
    ].copy()

    for _, row in strong_slow.iterrows():
        case_rows.append({
            "Country": row["Country"],
            "shock_id": shock_id,
            "baseline_year": row["baseline_year"],
            "case_type": "frontier_but_slow_recovery",
            "layer": row["layer"],
            "profile_code": row["profile_code"],
            "recovery_years": row["recovery_years"],
            "shock_median_recovery": shock_median_recovery,
            "interpretation": "Structurally undominated but recovered slower than the shock median; needs contextual explanation.",
        })

    weak_fast = group[
        group["layer"].eq(max_layer)
        & group["recovery_years"].notna()
        & (group["recovery_years"] < shock_median_recovery)
    ].copy()

    for _, row in weak_fast.iterrows():
        case_rows.append({
            "Country": row["Country"],
            "shock_id": shock_id,
            "baseline_year": row["baseline_year"],
            "case_type": "deep_layer_but_fast_recovery",
            "layer": row["layer"],
            "profile_code": row["profile_code"],
            "recovery_years": row["recovery_years"],
            "shock_median_recovery": shock_median_recovery,
            "interpretation": "Structurally weaker layer but recovered faster than the shock median; may reflect policy response or unmodeled factors.",
        })

recovery_interpretation_cases = pd.DataFrame(case_rows)

if recovery_interpretation_cases.empty:
    recovery_interpretation_cases = pd.DataFrame(columns=[
        "Country", "shock_id", "baseline_year", "case_type",
        "layer", "profile_code", "recovery_years",
        "shock_median_recovery", "interpretation",
    ])

save_table(
    recovery_interpretation_cases,
    "recovery_interpretation_cases.csv",
    "Recovery interpretation cases",
    "Mismatch cases where structural POSet position and GDP recovery outcome diverge.",
)

display(recovery_interpretation_cases)


Saved: recovery_interpretation_cases.csv


,Country,shock_id,baseline_year,case_type,layer,profile_code,recovery_years,shock_median_recovery,interpretation
0,DNK,2007,2007,frontier_but_slow_recovery,1,4-5-5-4-5,7.0000,5.0000,Structurally undominated but recovered slower ...
1,EST,2007,2007,frontier_but_slow_recovery,1,5-5-2-5-5,8.0000,5.0000,Structurally undominated but recovered slower ...
2,FIN,2007,2007,frontier_but_slow_recovery,1,3-2-5-5-3,8.0000,5.0000,Structurally undominated but recovered slower ...
3,SVN,2007,2007,frontier_but_slow_recovery,1,5-4-3-2-4,8.0000,5.0000,Structurally undominated but recovered slower ...
4,CZE,2019,2019,frontier_but_slow_recovery,1,5-5-4-1-4,2.0000,1.0000,Structurally undominated but recovered slower ...
5,DEU,2019,2019,frontier_but_slow_recovery,1,3-5-5-2-2,2.0000,1.0000,Structurally undominated but recovered slower ...


In [10]:

# ------------------------------------------------------
# TAKEAWAY TABLE
# ------------------------------------------------------

takeaway_rows = []

for shock_id in sorted(validation_dataset["shock_id"].unique()):
    frontier_cmp = profile_frontier_recovery_comparison[
        profile_frontier_recovery_comparison["shock_id"].eq(shock_id)
    ]

    trend = profile_layer_recovery_trend[
        profile_layer_recovery_trend["shock_id"].eq(shock_id)
    ]

    pareto_count = int(validation_dataset[
        validation_dataset["shock_id"].eq(shock_id)
        & validation_dataset["is_pareto_frontier"]
    ]["Country"].nunique())

    total_count = int(validation_dataset[
        validation_dataset["shock_id"].eq(shock_id)
    ]["Country"].nunique())

    if len(frontier_cmp):
        mean_diff = frontier_cmp["frontier_mean_minus_nonfrontier_mean_recovery"].iloc[0]
        median_diff = frontier_cmp["frontier_median_minus_nonfrontier_median_recovery"].iloc[0]
    else:
        mean_diff = np.nan
        median_diff = np.nan

    if len(trend):
        layer_corr = trend["layer_recovery_spearman_correlation"].iloc[0]
    else:
        layer_corr = np.nan

    if pd.notna(mean_diff):
        if mean_diff < 0:
            headline = "Frontier countries recovered faster on average."
        elif mean_diff > 0:
            headline = "Frontier countries recovered slower on average."
        else:
            headline = "Frontier and non-frontier countries had similar average recovery."
    else:
        headline = "Recovery comparison unavailable."

    takeaway_rows.append({
        "shock_id": shock_id,
        "countries": total_count,
        "frontier_countries": pareto_count,
        "frontier_mean_minus_nonfrontier_mean_recovery": mean_diff,
        "frontier_median_minus_nonfrontier_median_recovery": median_diff,
        "layer_recovery_spearman_correlation": layer_corr,
        "headline_takeaway": headline,
        "caution": "Validation is descriptive and does not establish causality.",
    })

recovery_validation_takeaway_table = pd.DataFrame(takeaway_rows)

working_main_profile_recovery_validation_summary = profile_recovery_validation_summary.copy()
working_main_profile_recovery_validation_summary["result_role"] = "main_profile_poset_validation"

save_table(
    recovery_validation_takeaway_table,
    "recovery_validation_takeaway_table.csv",
    "Recovery validation takeaway table",
    "Compact shock-level takeaways for recovery validation.",
)

save_table(
    working_main_profile_recovery_validation_summary,
    "working_main_profile_recovery_validation_summary.csv",
    "Working main profile recovery validation summary",
    "Main profile POSet recovery validation summary.",
)

display(recovery_validation_takeaway_table)


Saved: recovery_validation_takeaway_table.csv
Saved: working_main_profile_recovery_validation_summary.csv


,shock_id,countries,frontier_countries,frontier_mean_minus_nonfrontier_mean_recovery,frontier_median_minus_nonfrontier_median_recovery,layer_recovery_spearman_correlation,headline_takeaway,caution
0,2007,25,8,-0.8235,1.0000,0.0698,Frontier countries recovered faster on average.,Validation is descriptive and does not establi...
1,2019,35,13,-0.2632,0.0000,0.3095,Frontier countries recovered faster on average.,Validation is descriptive and does not establi...


In [11]:

# ------------------------------------------------------
# REPORT-READY RECOVERY VALIDATION PARAGRAPHS
# ------------------------------------------------------

def fmt_num(x, digits=2):
    if pd.isna(x):
        return "not available"
    return f"{x:.{digits}f}"

paragraph_rows = []

for _, row in recovery_validation_takeaway_table.iterrows():
    shock_id = row["shock_id"]

    paragraph_rows.append({
        "topic": f"Recovery validation summary {shock_id}",
        "report_text": (
            f"For the {shock_id} shock, the recovery validation compares countries' pre-shock POSet position "
            f"with their observed GDP recovery time. The main profile POSet contains {int(row['countries'])} "
            f"countries, of which {int(row['frontier_countries'])} belong to undominated frontier profiles. "
            f"The frontier-minus-non-frontier mean recovery difference is "
            f"{fmt_num(row['frontier_mean_minus_nonfrontier_mean_recovery'])} years, while the corresponding "
            f"median difference is {fmt_num(row['frontier_median_minus_nonfrontier_median_recovery'])} years. "
            f"This validation is descriptive and should not be interpreted as causal evidence."
        ),
    })

paragraph_rows.extend([
    {
        "topic": "Recovery validation role",
        "report_text": (
            "GDP recovery is used only after the POSet has been constructed. It is not part of the five ordering "
            "variables and therefore does not influence frontier status, layer assignment, or dominance relations."
        ),
    },
    {
        "topic": "Validation interpretation",
        "report_text": (
            "The validation step asks whether structurally stronger pre-shock profiles are associated with faster "
            "observed recovery. Mismatch cases are expected because recovery can also depend on policy response, "
            "shock exposure, sectoral composition, and other factors not included in the five structural variables."
        ),
    },
    {
        "topic": "No causal claim",
        "report_text": (
            "The analysis should be interpreted as structural and descriptive. A faster recovery among certain "
            "profiles supports the interpretability of the framework, but it does not prove that the selected "
            "structural variables caused the recovery pattern."
        ),
    },
])

report_ready_recovery_validation_paragraphs = pd.DataFrame(paragraph_rows)

methodological_decision_updates_recovery_validation = pd.DataFrame([
    {
        "decision": "Use GDP recovery only as validation",
        "choice": "Recovery_2007 and Recovery_2019 are external outcomes",
        "reason": "Recovery is an observed outcome after the shock, not a pre-shock capacity variable.",
        "risk_controlled": "Prevents outcome leakage into POSet ordering.",
    },
    {
        "decision": "Validate main profile POSet first",
        "choice": "Frontier/layer/profile comparison against recovery",
        "reason": "The main result is the 5-level profile POSet from Notebook 08.",
        "risk_controlled": "Keeps robustness checks separate from the primary empirical structure.",
    },
    {
        "decision": "Include mismatch cases",
        "choice": "frontier-but-slow and deep-layer-but-fast cases",
        "reason": "Mismatch cases improve interpretation and avoid overclaiming.",
        "risk_controlled": "Prevents treating POSet position as deterministic prediction.",
    },
])

save_table(
    report_ready_recovery_validation_paragraphs,
    "report_ready_recovery_validation_paragraphs.csv",
    "Report-ready recovery validation paragraphs",
    "Report-ready text for the recovery validation section.",
)

save_table(
    methodological_decision_updates_recovery_validation,
    "methodological_decision_updates_recovery_validation.csv",
    "Methodological decision updates recovery validation",
    "Decision-log entries for recovery validation.",
)

display(report_ready_recovery_validation_paragraphs)
display(methodological_decision_updates_recovery_validation)


Saved: report_ready_recovery_validation_paragraphs.csv
Saved: methodological_decision_updates_recovery_validation.csv


,topic,report_text
0,Recovery validation summary 2007,"For the 2007 shock, the recovery validation co..."
1,Recovery validation summary 2019,"For the 2019 shock, the recovery validation co..."
2,Recovery validation role,GDP recovery is used only after the POSet has ...
3,Validation interpretation,The validation step asks whether structurally ...
4,No causal claim,The analysis should be interpreted as structur...


,decision,choice,reason,risk_controlled
0,Use GDP recovery only as validation,Recovery_2007 and Recovery_2019 are external o...,Recovery is an observed outcome after the shoc...,Prevents outcome leakage into POSet ordering.
1,Validate main profile POSet first,Frontier/layer/profile comparison against reco...,The main result is the 5-level profile POSet f...,Keeps robustness checks separate from the prim...
2,Include mismatch cases,frontier-but-slow and deep-layer-but-fast cases,Mismatch cases improve interpretation and avoi...,Prevents treating POSet position as determinis...


In [12]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "11_Recovery_Validation_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    recovery_validation_takeaway_table.to_excel(writer, sheet_name="takeaways", index=False)
    profile_recovery_validation_summary.to_excel(writer, sheet_name="frontier_validation", index=False)
    profile_frontier_recovery_comparison.to_excel(writer, sheet_name="frontier_comparison", index=False)
    profile_layer_recovery_validation_summary.to_excel(writer, sheet_name="layer_validation", index=False)
    profile_layer_recovery_trend.to_excel(writer, sheet_name="layer_trend", index=False)
    profile_group_recovery_summary.to_excel(writer, sheet_name="profile_groups", index=False)
    pareto_recovery_validation_table.to_excel(writer, sheet_name="pareto_countries", index=False)
    epsilon_margin_recovery_validation_summary.to_excel(writer, sheet_name="epsilon_margin_valid", index=False)
    recovery_interpretation_cases.to_excel(writer, sheet_name="mismatch_cases", index=False)
    report_ready_recovery_validation_paragraphs.to_excel(writer, sheet_name="report_paragraphs", index=False)
    methodological_decision_updates_recovery_validation.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\11_Recovery_Validation_Audit.xlsx


In [13]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "recovery_validation_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "recovery_validation_table_inventory.csv", index=False)

expected_outputs = [
    "recovery_validation_country_dataset.csv",
    "profile_recovery_validation_summary.csv",
    "profile_frontier_recovery_comparison.csv",
    "pareto_recovery_validation_table.csv",
    "profile_layer_recovery_validation_summary.csv",
    "profile_layer_recovery_trend.csv",
    "profile_group_recovery_summary.csv",
    "epsilon_margin_recovery_validation_dataset.csv",
    "epsilon_margin_recovery_validation_summary.csv",
    "working_main_epsilon_margin_recovery_validation_summary.csv",
    "recovery_interpretation_cases.csv",
    "recovery_validation_takeaway_table.csv",
    "working_main_profile_recovery_validation_summary.csv",
    "report_ready_recovery_validation_paragraphs.csv",
    "methodological_decision_updates_recovery_validation.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("11 RECOVERY VALIDATION COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 11_Recovery_Validation_Audit.xlsx")
print("- recovery_validation_takeaway_table.csv")
print("- profile_recovery_validation_summary.csv")
print("- profile_layer_recovery_validation_summary.csv")
print("- recovery_interpretation_cases.csv")
print("- report_ready_recovery_validation_paragraphs.csv")

print("\nNext notebook:")
print("12_Sensitivity_Analysis_2002_5Var.ipynb")


11 RECOVERY VALIDATION COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,recovery_validation_country_dataset.csv,True,True,True
1,profile_recovery_validation_summary.csv,True,True,True
2,profile_frontier_recovery_comparison.csv,True,True,True
3,pareto_recovery_validation_table.csv,True,True,True
4,profile_layer_recovery_validation_summary.csv,True,True,True
5,profile_layer_recovery_trend.csv,True,True,True
6,profile_group_recovery_summary.csv,True,True,True
7,epsilon_margin_recovery_validation_dataset.csv,True,True,True
8,epsilon_margin_recovery_validation_summary.csv,True,True,True
9,working_main_epsilon_margin_recovery_validatio...,True,True,True



Key outputs to inspect/send:
- 11_Recovery_Validation_Audit.xlsx
- recovery_validation_takeaway_table.csv
- profile_recovery_validation_summary.csv
- profile_layer_recovery_validation_summary.csv
- recovery_interpretation_cases.csv
- report_ready_recovery_validation_paragraphs.csv

Next notebook:
12_Sensitivity_Analysis_2002_5Var.ipynb
